In [1]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import xgboost as xgb

In [2]:
df_sample = pd.read_csv('mid_project/sample_submission.csv')
df_test = pd.read_csv('mid_project/test_copy.csv')
df_train = pd.read_csv('mid_project/train_df.csv')

In [3]:
df_sample.shape, df_test.shape, df_train.shape

((48108, 2), (48108, 8), (50512, 9))

In [4]:
X = df_train.drop(['target'], axis=1)
y = df_train['target']
X.shape, y.shape

((50512, 8), (50512,))

In [5]:
def objective(trial):

    params = {
        'scale_pos_weight': 13.4,

        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),

        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'eval_metric': 'auc',
        'random_state': 4,
        'verbosity': 0,
        'nthread': -1
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=5)
    scores = []

    for fold, (train_index, val_index) in enumerate(skf.split(X, y)):
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]

        model = xgb.XGBClassifier(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict_proba(X_val)[:, 1]
        roc_auc = roc_auc_score(y_val, y_pred)
        scores.append(roc_auc)

    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40)

[I 2026-02-10 13:47:04,836] A new study created in memory with name: no-name-6105ccb2-7081-4b4d-a954-6a5f1a6f3289
[I 2026-02-10 13:47:06,888] Trial 0 finished with value: 0.8392024172571242 and parameters: {'min_child_weight': 10, 'subsample': 0.6887175058637764, 'colsample_bytree': 0.5617902612822253, 'reg_alpha': 0.03046923962330897, 'reg_lambda': 1.0301284863524152e-05, 'n_estimators': 903, 'learning_rate': 0.016884760988035236, 'max_depth': 4}. Best is trial 0 with value: 0.8392024172571242.
[I 2026-02-10 13:47:09,340] Trial 1 finished with value: 0.8243115818474233 and parameters: {'min_child_weight': 9, 'subsample': 0.7079773102223417, 'colsample_bytree': 0.958422060355925, 'reg_alpha': 0.01417732719441289, 'reg_lambda': 0.13162595767254798, 'n_estimators': 980, 'learning_rate': 0.032906731266256595, 'max_depth': 5}. Best is trial 0 with value: 0.8392024172571242.
[I 2026-02-10 13:47:12,986] Trial 2 finished with value: 0.8201411013470056 and parameters: {'min_child_weight': 9, '

In [6]:
best_params = study.best_params
best_params

{'min_child_weight': 5,
 'subsample': 0.7305545423130373,
 'colsample_bytree': 0.6105322462542344,
 'reg_alpha': 3.0927702453554105,
 'reg_lambda': 4.7953903983938994e-08,
 'n_estimators': 171,
 'learning_rate': 0.07097664010248834,
 'max_depth': 3}

In [7]:
print(f'ROC AUC score :{study.best_value:.4f}')

ROC AUC score :0.8414


In [8]:
xgb_model = xgb.XGBClassifier(**best_params, random_state=5)

In [9]:
xgb_model.fit(X, y)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.6105322462542344
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [10]:
y_pred_xgb = xgb_model.predict_proba(df_test)

In [11]:
df_sample['Predicted'] = y_pred_xgb
df_sample.head()

,Id,Predicted
0,1,0.883040
1,2,0.432501
2,3,0.984605
3,4,0.981977
4,5,0.603672


In [ ]:
df_sample.to_csv('submission.csv', index=False)

In [12]:
# import joblib
# joblib.dump(xgb_model, 'xgb_kfold.joblib')

['xgb_kfold.joblib']